In [0]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import numpy as np

In [0]:
ds = pd.read_csv('/Workspace/Users/w10168315@usm.edu/HonorsThesis/final_dataset.csv')

In [0]:
print(ds)

In [0]:
print(ds.columns)

### Removed rows with null/empty "text" values

In [0]:
# Set this to the name of your text column (e.g. 'text', 'comment', 'content')
text_col = 'text'

if text_col not in ds.columns:
    print(f"'{text_col}' not found. Available columns: {list(ds.columns)}")

else:
    before = len(ds)
    ds = ds[ds[text_col].notna() & (ds[text_col].astype(str).str.strip() != '')].reset_index(drop=True)
    print(f"Removed {before - len(ds)} rows with empty '{text_col}'. {len(ds)} rows remain.")

## Check the count of each column

In [0]:
print(ds['label'].isnull().sum())
print(ds['attack_type'].isnull().sum())
print(ds['is_synthetic'].isnull().sum())

## Check for duplicates and near-duplicates

This cell is a three-tier duplicate audit of your text column. The broader logic: you want to catch redundancy at increasing levels of "fuzziness" — first texts that are literally identical, then texts that are identical once you ignore cosmetic differences (case, punctuation, whitespace), and finally texts that are almost the same (small rewordings). For a jailbreak-classification thesis, this matters because duplicates inflate evaluation metrics and leak between train/test splits.

Section 1 — Exact duplicates (you've seen this part)
- Uses duplicated(keep=False) to flag every copy of any repeated text, counts the flagged rows, counts how many distinct texts are involved, and shows the top 10 most-repeated texts via groupby().size().

Section 2 — Duplicates after normalization
- def normalize(t): — defines a cleaning function that standardizes text before comparing:
- str(t).lower() — lowercases everything (so "Ignore ALL instructions" = "ignore all instructions").
- re.sub(r'[^a-z0-9\s]', ' ', t) — replaces every character that is not a lowercase letter, digit, or whitespace with a space. This strips punctuation and symbols.
- re.sub(r'\s+', ' ', t).strip() — collapses multiple spaces into one and trims the ends. Without this, replacing punctuation with spaces would create runs of spaces that make two identical texts look different.
- ds['_text_norm'] = ds[text_col].apply(normalize) — applies that function to every text, storing results in a temporary helper column.
- norm_dup_mask = ds['_text_norm'].duplicated(keep=False) — same duplicate detection as before, but on the normalized texts. This catches pairs like "Drop everything. Do X now" vs "drop everything do x now" — different raw strings, same content.
- The print + groupby block reports how many rows are redundant only after normalization. Comparing this number to Section 1's tells you how much duplication is "hidden" by formatting differences.

Section 3 — Near duplicates (TF-IDF + cosine similarity)
- Catches semantic near-copies: e.g., the same jailbreak template with a word or two changed.

- SAMPLE_SIZE = 2000 and the if len(ds) > SAMPLE_SIZE block — the comment explains why: cosine similarity compares every row against every other row, which is O(n²) — 2,000 rows means ~2 million comparisons; 50,000 rows would mean 1.25 billion and would grind to a halt. So if the dataset is big, it analyzes a random sample of 2,000 instead. random_state=42 makes the sample reproducible.

- TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5)) — converts each normalized text into a numeric vector. The key choices:

- char_wb means it analyzes character n-grams (not words), which is more robust to typos, misspellings, and word-level rewordings.

- (3, 5) means it uses all character sequences of 3, 4, and 5 characters. TF-IDF weights each n-gram by how distinctive it is — rare sequences count more than common ones.

- tfidf = vectorizer.fit_transform(sample['_text_norm']) — produces a sparse matrix: one row per text, one column per n-gram.

- sim_matrix = cosine_similarity(tfidf) — computes an n×n matrix where entry (i, j) is the cosine similarity between texts i and j: 1.0 = identical direction (very similar), 0.0 = nothing in common.

- np.fill_diagonal(sim_matrix, 0) — every text is 100% similar to itself, so the diagonal would dominate. Zeroing it removes self-matches.

- SIM_THRESHOLD = 0.85 — pairs scoring 0.85 or above are considered near duplicates. This is a tunable judgment call: higher = stricter, lower = more (possibly false) matches.

- The near_pairs = [...] list comprehension loops over all row pairs (i < j, avoiding double-counting and self-pairs) and keeps those meeting the threshold, along with their scores.

- The final loop prints the first 20 pairs with their similarity score and the first 100 characters of each text — a quick human spot-check before you decide to drop anything.


# INTERPRETATION OF RESULTS

**RESULT: 1,038 normalized = 1,014 exact-duplicate rows (all carried over) + 24 rows that are duplicates only once you ignore formatting.**

### Section 1 — Exact duplicates: 1,014 rows are redundant copies

- 1,014 rows are exact copies of 475 distinct texts — so on average each duplicated text appears ~2.1 times, and 1,014 − 475 = 539 rows are pure redundancy you could delete without losing any unique text.
- The top-10 table shows what kind of texts repeat, and it reveals your dataset's composition:

-   Flan-CoT style prompts ("Come up with a question and stream of consciousness reasoning...") — these are instruction-tuning data, repeated 4–8 times each.
-   Famous jailbreak templates — DAN, Developer Mode, RYX, and "Ignore all previous instructions..." — the classics that appear in nearly every public jailbreak corpus, which is exactly why they're duplicated: multiple source datasets carry copies of the same well-known prompts.

### Section 2 — Normalization adds little: formatting is not your problem
- 1,038 vs 1,014 — only 24 extra rows flagged after ignoring case/punctuation/whitespace. Cosmetic-variant duplication is minor.
- The biggest normalized cluster (count 10) is the Developer Mode prompt — the DAN-Mode and Developer-Mode variants you saw as separate raw texts collapsed into shared clusters here, but few fully merged.

### Section 3 — Near duplicates: 220 pairs, but read them critically
- The sample found 220 pairs ≥0.85 similarity. They fall into three very different categories:

- Genuine near-duplicates — worth acting on. E.g., the four HealthBot prompts (rows 5, 241, 791, 1596) are one template with minor rewordings ("your user's" vs "your users'" vs "provide me with"). 
- Same for the hex-encoded prompt cluster (rows 56/1557 are identical intent; 1901 drops the instruction and is just raw hex) and "Disclose/Provide confidential information..." (0.880).
- False positives from a data-quality artifact — this is the big insight: Row 47 (I   a m   a   s t u d e n t   a n d...) matches 13+ completely unrelated prompts at 0.86–0.94. Why? Those texts are spelled with spaces between every character/word. Character 3–5-grams of spaced text ("a m", "s t u", " t h") are generic, so every spaced-out text looks similar to every other. These pairs are noise, not near-duplicates — and they also flag that some rows in your dataset have mangled spacing.
- Benign content overlap — row 37's sports table is Flan tabular data; table headers share n-grams with other tables. Harmless.

## What to do next (in order)
- Check for label conflicts first. Before dropping anything, group the exact duplicates by text and check whether all copies share the same label/attack_type. The same prompt labeled both jailbreak and benign would be a labeling inconsistency — a much more serious thesis finding than redundancy.
- Deduplicate on the normalized text before any train/test split (drop copies, keep one). 539+ redundant rows inflate metrics and risk the same text landing in both train and test.
- Don't blind-drop the 220 near-dup pairs — most are sample artifacts or spaced-text noise. Review genuine clusters like HealthBot manually, and note in your thesis that dedup was exact + normalized only, with near-dup pairs reviewed.
- Fix or flag the spaced-text rows — count how many rows match that single-letter-spacing pattern; either repair them (collapse the extra spaces) or note them as a known source artifact.


In [0]:
# text_col = 'text'

# # --- 1. Exact duplicates (raw text) ---

# exact_dup_mask = ds[text_col].duplicated(keep=False) # For each row, returns True if its text appears >1 anywhere in ds; keep=false means it also counts the original.

# print(f"Exact duplicate rows (including originals): {exact_dup_mask.sum()}") # Counts all rows whose text is duplicated, counting every copy

# print(f"Distinct duplicated texts: {ds.loc[exact_dup_mask, text_col].nunique()}") # Among flagged rows, counts how many unique text values are actually involved

# print(ds[exact_dup_mask].groupby(text_col).size().sort_values(ascending=False).head(10)) # Groups duplicated rows by their text & times each appears, and show the top 10 ones



# # --- 2. Duplicates after normalization (whitespace, case, punctuation) ---
# def normalize(t):
#     t = str(t).lower() #lowercase everything
#     t = re.sub(r'[^a-z0-9\s]', ' ', t) #replaces characters (not lowercase letter, digit, or whitespace) with a space
#     t = re.sub(r'\s+', ' ', t).strip() #collapses multiple spaces into one and trims the ends
#     return t

# ds['_text_norm'] = ds[text_col].apply(normalize) #applies normalize function to every text, storing results in a temporary helper column

# norm_dup_mask = ds['_text_norm'].duplicated(keep=False) #duplicate detection on the normalized texts

# print(f"\nNormalized duplicate rows: {norm_dup_mask.sum()}") #total count

# print(ds[norm_dup_mask].groupby('_text_norm').size().sort_values(ascending=False).head(10)) #groupBy



# # --- 3. Near duplicates (TF-IDF + cosine similarity) ---
# SAMPLE_SIZE = 2000
# if len(ds) > SAMPLE_SIZE: #if ds big, anlyz random sampl 2,000-42(reproducible)
#     sample = ds.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
#     print(f"\nDataset is large; comparing a random sample of {SAMPLE_SIZE} rows for near duplicates.")

# else:
#     sample = ds.reset_index(drop=True)

# vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5)) #converts each normalized text into a numeric vector

# tfidf = vectorizer.fit_transform(sample['_text_norm'])

# sim_matrix = cosine_similarity(tfidf) # n×n matrix, where (i, j) is cosine similarity betn texts i & j: 1.0(identical direction) & 0.0 = nothing common.


# np.fill_diagonal(sim_matrix, 0)

# SIM_THRESHOLD = 0.85 #pairs scoring >= 0.85 are near duplicates

# # list comprehension loops over all row pairs & keep those meeting threshold, along with scores.
# near_pairs = [
#     (i, j, sim_matrix[i, j])
#     for i in range(len(sample))
#     for j in range(i + 1, len(sample))
#     if sim_matrix[i, j] >= SIM_THRESHOLD
# ]

# # prints the 1st 20 pairs with similarity score & first 100 char of each text
# print(f"\nNear-duplicate pairs (cosine similarity >= {SIM_THRESHOLD}): {len(near_pairs)}")

# for i, j, score in near_pairs[:20]:
#     print(f"\nSim: {score:.3f}")
#     print(f"  Row {i}: {sample.loc[i, text_col][:100]}")
#     print(f"  Row {j}: {sample.loc[j, text_col][:100]}")

# # Drop helper column
# ds = ds.drop(columns=['_text_norm']) # removes temp column

## View exact duplicates

In [0]:
# Find all rows with duplicate text (including originals)
exact_duplicates = ds[ds.duplicated(subset=['text'], keep=False)]

# Sort by text to group identical prompts together
exact_duplicates = exact_duplicates.sort_values(by='text')

print("Total duplicate rows:", len(exact_duplicates))
print("Unique duplicated texts:", exact_duplicates['text'].nunique())

display(exact_duplicates)

## Remove exact duplicates

In [0]:
# Remove duplicate texts, keeping the first occurrence
ds2 = ds.drop_duplicates(subset=['text'], keep='first').copy()

# Reset row indices
ds2.reset_index(drop=True, inplace=True)

# Check how many rows were removed
print("Original rows:", len(ds))
print("Cleaned rows:", len(ds2))
print("Duplicates removed:", len(ds) - len(ds2))

- 1,014: Rows containing text that appears at least twice, including the original occurrences.
- 475: There are 475 distinct texts that appear multiple times.
- 539: Extra copies deleted while keeping one occurrence of each text.
- 475: One copy of each of the 475 repeated texts remains in your cleaned dataset.

## TEST: Check conflicts in Label

In [0]:
# Find texts assigned multiple labels
conflicts = ds2.groupby('text')['label'].nunique()

conflicts = conflicts[conflicts > 1]
print(conflicts)
print("Texts with conflicting labels:", len(conflicts))

## View Normalized Duplicates

In [0]:
# Function to normalize text
def normalize(t):
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Create a normalized text column
ds2['_text_norm'] = ds2['text'].apply(normalize)

# Find all normalized duplicates, including originals
normalized_duplicates = ds2[
    ds2.duplicated(subset=['_text_norm'], keep=False)
].copy()

# Sort to group matching texts together
normalized_duplicates = normalized_duplicates.sort_values('_text_norm')

# Display all columns and complete text
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

display(normalized_duplicates)

## Remove Normalized Duplicates

In [0]:
# Remove normalized duplicates, keeping one row per group
ds3 = ds2.drop_duplicates(
    subset=['_text_norm'],
    keep='first'
).copy()

# Reset row indices
ds3.reset_index(drop=True, inplace=True)

# Check how many normalized duplicates remain
print("Remaining normalized duplicates:",
      ds3.duplicated(subset=['_text_norm']).sum())

In [0]:
print(len(ds3))

## View character-spaced prompts
- Note: They are not duplicates. Preserve them as they are for the model training.

In [0]:
# Detect sequences of at least 5 single letters separated by spaces
def is_character_spaced(text):
    pattern = r'(?<!\w)(?:[A-Za-z]\s+){4,}[A-Za-z](?!\w)'
    return bool(re.search(pattern, str(text)))

# Find character-spaced prompts
spaced_prompts = ds3[
    ds3['text'].apply(is_character_spaced)
].copy()

# Display results
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print("Total character-spaced prompts:", len(spaced_prompts))
display(spaced_prompts)

## View near-duplicate prompts (similarity ≥ 0.90)

In [0]:
# Settings
SAMPLE_SIZE = 2000
SIM_THRESHOLD = 0.90

# Select a reproducible random sample
sample = (
    ds3.sample(n=SAMPLE_SIZE, random_state=42)
    if len(ds3) > SAMPLE_SIZE
    else ds3.copy()
).reset_index(drop=False)

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5)
)

# Use the normalized text column created earlier
tfidf = vectorizer.fit_transform(
    sample['_text_norm'].fillna('')
)

# Calculate similarity
sim_matrix = cosine_similarity(tfidf)

# Ignore self-comparisons
np.fill_diagonal(sim_matrix, 0)

# Find pairs with similarity >= 0.90
rows, cols = np.where(
    np.triu(sim_matrix, k=1) >= SIM_THRESHOLD
)

# Create a DataFrame for easy inspection
near_duplicates = pd.DataFrame({
    'row_1': sample.loc[rows, 'index'].values,
    'text_1': sample.loc[rows, 'text'].values,
    'row_2': sample.loc[cols, 'index'].values,
    'text_2': sample.loc[cols, 'text'].values,
    'similarity': sim_matrix[rows, cols]
})

# Sort by highest similarity
near_duplicates = near_duplicates.sort_values(
    'similarity', ascending=False
).reset_index(drop=True)

# Display results
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print("Total near-duplicate pairs:", len(near_duplicates))
display(near_duplicates)

## Remove near-duplicates (similarity >= 0.90)

In [0]:
# Each pair connects two similar rows
parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# Group related near-duplicate rows
for _, row in near_duplicates.iterrows():
    union(int(row['row_1']), int(row['row_2']))

# Collect groups
groups = {}
for idx in parent:
    root = find(idx)
    groups.setdefault(root, []).append(idx)

# Keep one row from each group
rows_to_remove = []

for members in groups.values():
    members = sorted(members)
    rows_to_remove.extend(members[1:])

# Remove redundant rows
before = len(ds3)

ds4 = ds3.drop(
    index=rows_to_remove
).reset_index(drop=True)

print("Near-duplicate groups:", len(groups))
print("Rows removed:", before - len(ds4))
print("Remaining rows:", len(ds4))

- Your 75 groups originally contained: 75 + 102 = 177 rows
- You kept one row per group: 177 - 75 = 102 rows removed

In [0]:
# Save the cleaned dataset
ds4.to_csv("final_CLEANED_dataset.csv", index=False)

print("Dataset saved successfully!")
print("Total rows:", len(ds4))
print("Total columns:", len(ds4.columns))